# wmam on Colab (GPU)

Runtime → Change runtime type → **GPU** before running. Every training command takes only a
**setup file** (input) and an **output path**:

```
python -m kinkchain.make_setup --config configs/example_cyclic.json --out setups/example_cyclic.setup.pkl   # freeze data + init + kinks
python -m kinkchain.run_setup  --setup setups/example_cyclic.setup.pkl         --out runs/example_bulk.npz       # train
```

Code comes from GitHub (`git clone` / `git pull`); outputs go to `runs/` (not committed) — here
optionally a Google-Drive folder so they survive the session.

## 1. GPU check

In [ ]:
# run in a subprocess: a JAX client inside the notebook process would grab most of the GPU memory
!nvidia-smi -L
!python -c "import jax; print('jax', jax.__version__, jax.devices())"

## 2. Code from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/MJ-Zu/wmam.git"   # public: no login needed
BRANCH   = "main"
WORKDIR  = "/content/wmam"

if not os.path.isdir(WORKDIR):
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} pull --ff-only       # pick up what you pushed since the last session
%cd {WORKDIR}
!git log --oneline -3

In [ ]:
# jax comes with Colab (CUDA build on a GPU runtime); the rest is small
!pip install -q flax optax
# if the GPU check above showed only CpuDevice:  !pip install -q -U "jax[cuda12]"

## 3. Where the outputs go (optional: Google Drive)

`runs/` becomes a link to a Drive folder, so every output written by `--out runs/...` lands on Drive
directly and survives a disconnected runtime. Set `USE_DRIVE = False` to keep them on the
(temporary) Colab disk and download at the end.

In [ ]:
import os, shutil

USE_DRIVE  = True
DRIVE_RUNS = "/content/drive/MyDrive/wmam_runs"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_RUNS, exist_ok=True)
    if os.path.isdir("runs") and not os.path.islink("runs"):
        shutil.copytree("runs", DRIVE_RUNS, dirs_exist_ok=True)
        shutil.rmtree("runs")
    if not os.path.lexists("runs"):
        os.symlink(DRIVE_RUNS, "runs")
os.makedirs("runs", exist_ok=True)
os.makedirs("setups", exist_ok=True)
print(os.path.realpath("runs"))

## 4. Look at a config

A config JSON is `{"cfg": {...}, "run": {...}}`: `cfg` = WMAMConfig fields (the rest are defaults),
`run` = `mode` (`cyclic` | `single_tau` | `refine` | `further`), `grad_clip`, `stride`, ...
Edit a file under `configs/` (or push the change to GitHub and `git pull`), or just edit the dict
here and write it out.

In [ ]:
import json

CONFIG = "configs/example_cyclic.json"     # try configs/example_utkface_cyclic.json for the 64x64 faces
print(json.dumps(json.load(open(CONFIG)), indent=1))

Memory: the per-step weight trajectory takes `(steps + 1) × n_parameters × 4` bytes on the GPU
(64 channels, 100 000 steps ≈ 14 GiB). Reduce it with `"stride": k` (cyclic modes) or `"log_every": k`
(single_tau) in the config — only every k-th step is recorded, *inside* the training loop.

## 5. Freeze the setup, then train

In [ ]:
!python -m kinkchain.make_setup --config {CONFIG} --out setups/example_cyclic.setup.pkl
!python -m kinkchain.make_setup --show setups/example_cyclic.setup.pkl

In [ ]:
# stage 1 (cyclic: cold + bulk cycles)
!python -m kinkchain.run_setup --setup setups/example_cyclic.setup.pkl --out runs/example_bulk.npz

Next stages are small JSONs layered on the frozen setup (`"base": "setups/example_cyclic.setup.pkl"`), see
`configs/example_stage2_further.json` and `configs/example_stage3_refine.json` — their
`run.load` names the previous output. (`scripts/cyclic_pipeline.sh` runs all three in a row.)

In [ ]:
# !python -m kinkchain.run_setup --setup configs/example_stage2_further.json --out runs/example_further.npz
# !python -m kinkchain.run_setup --setup configs/example_stage3_refine.json  --out runs/example_refine.npz

Single fixed τ (bulk → refine) uses a `single_tau` config:
`!python -m kinkchain.run_setup --setup configs/example_single_tau.json --out runs/example_single_tau.npz`

## 6. Outputs

In [ ]:
!ls -lh runs

In [ ]:
# without Drive: download everything in runs/ as one zip
if not USE_DRIVE:
    shutil.make_archive("/content/wmam_runs", "zip", "runs")
    from google.colab import files
    files.download("/content/wmam_runs.zip")